# Extract CLIP Features


In [1]:
!pip install -q open_clip_torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [2]:
from pathlib import Path

import open_clip
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


In [3]:
CSV_PATH = "/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv"
DEEPFAKEBENCH_ROOT = "/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench"
FFPP_TEST_ROOT = "/kaggle/input/datasets/vohoanghoavien/ff-color-contrast-5/color_contrast"
CELEB_TEST_ROOT = "/kaggle/input/datasets/vohoanghoavien/celebdfv1-color-contrast-5/processed_output/color_contrast/level_5"

DATASET_NAME = "FaceForensics++"  # "FaceForensics++" or "Celeb-DF-v1"
USE_TEST_PATH = True              # True nếu muốn dùng color-contrast test path
OUTPUT_PATH = "/kaggle/working/ffpp-color-constrast-5.pt"

BATCH_SIZE = 64
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cuda'

In [4]:
def fix_path(path):
    return str(path).replace("../input/deepfakebench", DEEPFAKEBENCH_ROOT)


def to_test_path(path):
    path = str(path)
    if DATASET_NAME == "FaceForensics++":
        return path.replace(DEEPFAKEBENCH_ROOT, FFPP_TEST_ROOT)
    if DATASET_NAME == "Celeb-DF-v1":
        return path.replace(f"{DEEPFAKEBENCH_ROOT}/Celeb-DF-v1", CELEB_TEST_ROOT)
    return path


df = pd.read_csv(CSV_PATH)
df = df[df["datasetname"] == DATASET_NAME].copy()
df["label_num"] = df["label"].map({"REAL": 0, "FAKE": 1}).astype(int)
df["imagepath_fixed"] = df["imagepath"].apply(fix_path)

if USE_TEST_PATH:
    df["imagepath_fixed"] = df["imagepath_fixed"].apply(to_test_path)

df = df.reset_index(drop=True)

print(df.shape)
print(df["label_num"].value_counts())
df.head()


(513568, 7)
label_num
1    470878
0     42690
Name: count, dtype: int64


,imagepath,original_width,original_height,label,datasetname,label_num,imagepath_fixed
0,../input/deepfakebench/FaceForensics++/origina...,256,256,REAL,FaceForensics++,0,/kaggle/input/datasets/vohoanghoavien/ff-color...
1,../input/deepfakebench/FaceForensics++/origina...,256,256,REAL,FaceForensics++,0,/kaggle/input/datasets/vohoanghoavien/ff-color...
2,../input/deepfakebench/FaceForensics++/origina...,256,256,REAL,FaceForensics++,0,/kaggle/input/datasets/vohoanghoavien/ff-color...
3,../input/deepfakebench/FaceForensics++/origina...,256,256,REAL,FaceForensics++,0,/kaggle/input/datasets/vohoanghoavien/ff-color...
4,../input/deepfakebench/FaceForensics++/origina...,256,256,REAL,FaceForensics++,0,/kaggle/input/datasets/vohoanghoavien/ff-color...


In [5]:
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="openai",
)

clip_model = clip_model.to(DEVICE).eval()


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [6]:
class ImageDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["imagepath_fixed"]).convert("RGB")
        image = preprocess(image)
        label = int(row["label_num"])
        return image, label


loader = DataLoader(
    ImageDataset(df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)


In [7]:
features = []
labels = []

with torch.no_grad():
    for images, batch_labels in tqdm(loader):
        images = images.to(DEVICE, non_blocking=True)
        batch_features = clip_model.encode_image(images)
        batch_features = F.normalize(batch_features, dim=-1)

        features.append(batch_features.cpu())
        labels.append(batch_labels.cpu())

features = torch.cat(features)
labels = torch.cat(labels)

print(features.shape)
print(labels.shape)


100%|██████████| 8025/8025 [6:17:32<00:00,  2.82s/it]


torch.Size([513568, 768])
torch.Size([513568])


In [8]:
torch.save(
    {
        "features": features,
        "labels": labels,
        "dataset_name": DATASET_NAME,
        "use_test_path": USE_TEST_PATH,
        "clip_model": "ViT-L-14/openai",
    },
    OUTPUT_PATH,
)

print("saved to:", OUTPUT_PATH)


saved to: /kaggle/working/ffpp-color-constrast-5.pt
